In [0]:
columns = ["emp_id", "name", "department", "status", "salary"]
data = [
    (101, "Amit", "Engineering", "Full-Time", 90000),
    (102, "Neha", "Engineering", "Contractor", 60000),
    (103, "Rahul", "Engineering", "Full-Time", 110000),
    (104, "Pooja", "Marketing", "Full-Time", 75000),
    (105, "Vikram", "Marketing", "Contractor", 50000),
    (106, "Siddharth", "HR", "Full-Time", 65000) # HR has 0 contractors
]

df_emp = spark.createDataFrame(data, columns)
df_emp.show()

### pivot

In [0]:
df_emp.groupBy('department').pivot('status', ['Full-Time', 'Contractor']).sum('salary').show()

In [0]:
pv_df = df_emp.groupBy('status').pivot('department', ['Engineering','Marketing', 'HR']).sum('salary')
pv_df.show()

In [0]:
# unpivote the data
pv_df.selectExpr(
    "status",
    "stack(3, 'Engineering', Engineering, 'Marketing', Marketing, 'HR', HR) as (department, salary)"
).show()

In [0]:
# adding extra columns while unpivoting the data
pv_df.selectExpr(
    "status",
    "stack(3, 'Engineering', Engineering, CAST(Engineering*1.25 AS DOUBLE), 'Marketing', Marketing, CAST(Marketing*5.0 AS DOUBLE), 'HR', HR, CAST(HR*3.5 AS DOUBLE)) as (department, salary, inc_sal)"
).show()


### 3: Collection & List Aggregations (Non-Scalar Outputs)Scenario: 
- Gathering separate column row strings or values into a consolidated array list or unique set assigned to each specific group identifier.

In [0]:
from pyspark.sql.functions import collect_list, collect_set
columns = ["ticket_id", "agent_id", "assigned_tag"]
data = [
    ("T1", "A101", "Billing"), ("T1", "A101", "Refund"), ("T1", "A101", "Billing"),
    ("T2", "A102", "Technical"), ("T2", "A102", "Setup")
]
df = spark.createDataFrame(data, columns)
df.display()
# Solution: collect_list keeps duplicates; collect_set drops duplicates
df_result = df.groupBy('ticket_id') \
    .agg(
        collect_list("assigned_tag").alias("all_tags"),
        collect_set("assigned_tag").alias("unique_tags")
    )
df_result.show(truncate=False)


### 4: String and Conditional ConcatenationsScenario: 
- Merging discrete text fields into a single uniform string representation segregated by a designated separator element matching each key.

In [0]:
from pyspark.sql.functions import concat_ws, collect_list
columns = ["order_id", "item_name"]
data = [
    ("O10", "Laptop"), ("O10", "Mouse"), ("O10", "Keyboard"),
    ("O11", "Monitor"), ("O11", "HDMI Cable")
]
df = spark.createDataFrame(data, columns)
df.display()
# Solution: Combining multiple strings with a structured delimiter format
df_result = df.groupBy("order_id") \
    .agg(concat_ws(" -> ", collect_list("item_name")).alias("manifest_summary"))

df_result.show(truncate=False)
